# PhilIRI Bronze → Silver

This notebook audits the transformation of raw PhilIRI CSV exports into harmonized silver parquets.

PhilIRI is more complex than CRLA because:
1. **Two key stages**: KS2 (Elementary G4–G6) and KS3 (Secondary G7–G10) with different schemas
2. **Two assessment designs**: BoSY has a 7-level scale (Grade Ready + 3LD/2LD × F/I/Ind); EoSY only reports 3 levels (Frustration/Instructional/Independent) for reassessed students
3. **Naming inconsistency**: KS3 uses `{G} {lang} {level} 2Level` while KS2 uses `{G} 2LD {lang} {level}`
4. **Structural differences across years**: KS3 EoSY 2024-25 has one Assessed per grade; 2025-26 has per-language Assessed
5. **Data quality issues**: duplicate column, wrong enrolled header

**This notebook covers bronze → silver only.** Gold integration of PhilIRI indicators (EMD, proficiency levels) is a planned future step.

In [1]:
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'modules'))

from philiri_preprocessing import (
    PHILIRI_BRONZE_DIR, PHILIRI_SILVER_DIR,
    resolve_philiri_files,
    load_philiri_file,
    _normalize_philiri_columns,
    _drop_duplicates_columns,
    KS2_BOSY_COLUMNS, KS3_BOSY_COLUMNS,
    KS2_EOSY_COLUMNS, KS3_EOSY_COLUMNS_PER_GRADE, KS3_EOSY_COLUMNS_PER_LANG,
    METADATA_COLUMNS, KS2_GRADES, KS3_GRADES, LANGUAGES, BOSY_LEVELS_PER_LANG, EOSY_LEVELS_PER_LANG,
)

print('Imports OK')
print(f'Bronze dir: {PROJECT_ROOT / PHILIRI_BRONZE_DIR}')

Imports OK
Bronze dir: /workspace/project_crla/data/bronze/philiri


---
## 1. File Discovery

PhilIRI filenames contain non-breaking spaces (U+00A0), so we use glob patterns with leading wildcards. Regular `os.path.join` and string matching fail silently.

In [2]:
# Show actual filenames with byte-level inspection
bronze_dir = PROJECT_ROOT / PHILIRI_BRONZE_DIR

print('Actual filenames in bronze/philiri/ (with non-breaking space shown as ·):\n')
for fpath in sorted(bronze_dir.glob('*.csv')):
    # Replace U+00A0 with · for visibility
    display_name = fpath.name.replace('\u00a0', '·')
    size_kb = fpath.stat().st_size / 1000
    nbsps = fpath.name.count('\u00a0')
    print(f'  {display_name}  ({size_kb:.0f} KB, {nbsps} NBSP)')

Actual filenames in bronze/philiri/ (with non-breaking space shown as ·):

  Phil-IRI Results SY 2024-25 Archive_Secondary - BoSY_Table.csv  (2337 KB, 0 NBSP)
  Phil-IRI Results SY 2024-25 Archive_Secondary - EoSY_Table.csv  (1886 KB, 0 NBSP)
  Phil-IRI·KS2 National Dashboard_Elementary - BoSY 2025-26_Table.csv  (6592 KB, 1 NBSP)
  Phil-IRI·KS2 National Dashboard_Elementary - EoSY 2025-26_Table.csv  (5574 KB, 1 NBSP)
  Phil-IRI·KS3 National Dashboard_Secondary - BoSY 2025-26_Table.csv  (2668 KB, 1 NBSP)
  Phil-IRI·KS3 National Dashboard_Secondary - EoSY 2025-26_Table.csv  (2083 KB, 1 NBSP)
  Phil-IRI·Results SY 2024-25 Archive·Elementary - BoSY_Table.csv  (6020 KB, 2 NBSP)
  Phil-IRI·Results SY 2024-25 Archive·Elementary - EoSY_Table.csv  (5210 KB, 2 NBSP)


In [3]:
# Test that resolve_philiri_files() correctly maps to all 8 files
file_map = resolve_philiri_files(str(bronze_dir))

print(f'Files found: {len(file_map)} of 8 expected\n')
for (ks, sy, period), path in sorted(file_map.items()):
    df_peek = pd.read_csv(path, nrows=0, low_memory=False)
    print(f'  {ks:4s}  {sy}  {period:4s}  →  {len(df_peek.columns)} cols  |  {Path(path).name[:50]}...')

Files found: 8 of 8 expected

  ks2   2024-25  BoSY  →  52 cols  |  Phil-IRI KS2 National Dashboard_Elementary - BoSY ...
  ks2   2024-25  EoSY  →  36 cols  |  Phil-IRI KS2 National Dashboard_Elementary - EoSY ...
  ks2   2025-26  BoSY  →  52 cols  |  Phil-IRI KS2 National Dashboard_Elementary - BoSY ...
  ks2   2025-26  EoSY  →  36 cols  |  Phil-IRI KS2 National Dashboard_Elementary - EoSY ...
  ks3   2024-25  BoSY  →  71 cols  |  Phil-IRI Results SY 2024-25 Archive_Secondary - Bo...
  ks3   2024-25  EoSY  →  43 cols  |  Phil-IRI Results SY 2024-25 Archive_Secondary - Eo...
  ks3   2025-26  BoSY  →  71 cols  |  Phil-IRI KS3 National Dashboard_Secondary - BoSY 2...
  ks3   2025-26  EoSY  →  45 cols  |  Phil-IRI KS3 National Dashboard_Secondary - EoSY 2...


---
## 2. Schema Deep-Dive: BoSY

### 2a. KS2 BoSY (G4–G6)

The KS2 BoSY format uses a single `{G} Assessed` count per grade, then per-language breakdown:

```
G4 Assessed
G4 Fil Grade Ready
G4 2LD Fil Frustration / Instructional / Independent
G4 3LD Fil Frustration / Instructional / Independent
G4 Eng Grade Ready
G4 2LD Eng Frustration / Instructional / Independent
G4 3LD Eng Frustration / Instructional / Independent
```

**Reading this**: A student tagged as `2LD` is reading 2 levels below grade level; `3LD` is 3 levels below. `Grade Ready` means the student is at grade level at the start of the year.

In [4]:
# Show KS2 BoSY canonical columns for G4
print('Canonical KS2 BoSY columns for G4:')
g4_cols = [c for c in KS2_BOSY_COLUMNS if c.startswith('G4')]
for c in g4_cols:
    print(f'  {c}')
print(f'\nTotal canonical KS2 BoSY cols: {len(KS2_BOSY_COLUMNS)}')
print('(= 3 grades × (1 Assessed + 2 langs × 7 items) = 3 × 15 = 45)')

Canonical KS2 BoSY columns for G4:
  G4 Assessed
  G4 Fil Grade Ready
  G4 2LD Fil Frustration
  G4 2LD Fil Instructional
  G4 2LD Fil Independent
  G4 3LD Fil Frustration
  G4 3LD Fil Instructional
  G4 3LD Fil Independent
  G4 Eng Grade Ready
  G4 2LD Eng Frustration
  G4 2LD Eng Instructional
  G4 2LD Eng Independent
  G4 3LD Eng Frustration
  G4 3LD Eng Instructional
  G4 3LD Eng Independent

Total canonical KS2 BoSY cols: 45
(= 3 grades × (1 Assessed + 2 langs × 7 items) = 3 × 15 = 45)


In [5]:
# Read actual KS2 BoSY 2024-25 and compare raw vs canonical
path_ks2_bosy_2425 = file_map[('ks2', '2024-25', 'BoSY')]
df_raw = pd.read_csv(path_ks2_bosy_2425, nrows=3, low_memory=False)

print(f'Raw KS2 BoSY 2024-25: {len(df_raw.columns)} columns')
print('\nRaw column names (first 20):')
for i, c in enumerate(df_raw.columns[:20]):
    print(f'  [{i:2d}] {repr(c)}')

Raw KS2 BoSY 2024-25: 52 columns

Raw column names (first 20):
  [ 0] 'Region'
  [ 1] 'Division'
  [ 2] 'District'
  [ 3] 'School ID'
  [ 4] 'School Name'
  [ 5] 'Total Enrolled (G4-G6)'
  [ 6] 'Total Assessed (G4-G6)'
  [ 7] 'G4 Assessed'
  [ 8] 'G4 Fil Grade Ready'
  [ 9] 'G4 2LD Fil Frustration'
  [10] 'G4 2LD Fil Instructional'
  [11] 'G4 2LD Fil Independent'
  [12] 'G4 3LD Fil Frustration'
  [13] 'G4 3LD Fil Instructional'
  [14] 'G4 3LD Fil Independent'
  [15] 'G4 Eng Grade Ready'
  [16] 'G4 2LD Eng Frustration'
  [17] 'G4 2LD Eng Instructional'
  [18] 'G4 2LD Eng Independent'
  [19] 'G4 3LD Eng Frustration'


In [6]:
# ISSUE: Duplicate column in KS2 2024-25 BoSY
# G5 3LD Fil Frustration appears TWICE in the raw file.
# Pandas auto-renames the second to "G5 3LD Fil Frustration.1"
df_full_raw = pd.read_csv(path_ks2_bosy_2425, low_memory=False)

# Find any .N-suffixed columns
suffix_cols = [c for c in df_full_raw.columns if re.search(r'\.\d+$', c)]
print(f'Pandas-renamed duplicate columns in KS2 BoSY 2024-25:')
for c in suffix_cols:
    base = re.sub(r'\.\d+$', '', c)
    print(f'  {repr(c)}  →  base: {repr(base)}')
    # Show both values for a sample row
    if c in df_full_raw.columns and base in df_full_raw.columns:
        print(f'    Sample row 0: original={df_full_raw[base].iloc[0]}, duplicate={df_full_raw[c].iloc[0]}')

print()
print('Handling: _drop_duplicates_columns() keeps the first occurrence and drops .N variants.')

Pandas-renamed duplicate columns in KS2 BoSY 2024-25:

Handling: _drop_duplicates_columns() keeps the first occurrence and drops .N variants.


### 2b. KS3 BoSY (G7–G10)

KS3 uses a different column naming convention. The per-language Assessed columns are *included* (unlike KS2), and the level suffixes are reversed:

| KS2 format | KS3 format | Canonical (normalized) |
|---|---|---|
| `G4 2LD Fil Frustration` | `G7 Fil Frustration 2Level` | `G7 2LD Fil Frustration` |
| `G4 3LD Fil Frustration` | `G7 Fil Frustration 3Level` | `G7 3LD Fil Frustration` |
| `G4 Assessed` | `G7 Fil Assessed` + `G7 Eng Assessed` | same |

In [7]:
path_ks3_bosy_2425 = file_map[('ks3', '2024-25', 'BoSY')]
df_ks3_raw = pd.read_csv(path_ks3_bosy_2425, nrows=3, low_memory=False)

# Find the '2Level' columns
two_level_cols = [c for c in df_ks3_raw.columns if '2Level' in c]
print(f'Raw KS3 BoSY 2024-25: {len(df_ks3_raw.columns)} columns')
print(f'Columns containing "2Level": {len(two_level_cols)}')
print('\nExamples of 2Level columns:')
for c in two_level_cols[:6]:
    print(f'  Raw:        {repr(c)}')
    from philiri_preprocessing import _normalize_philiri_columns
    df_tmp = pd.DataFrame(columns=[c])
    norm = _normalize_philiri_columns(df_tmp).columns[0]
    print(f'  Normalized: {repr(norm)}')
    print()

Raw KS3 BoSY 2024-25: 71 columns
Columns containing "2Level": 24

Examples of 2Level columns:
  Raw:        'G7 Fil Frustration 2Level'
  Normalized: 'G7 2LD Fil Frustration'

  Raw:        'G7 Fil Instructional 2Level'
  Normalized: 'G7 2LD Fil Instructional'

  Raw:        'G7 Fil Independent 2Level'
  Normalized: 'G7 2LD Fil Independent'

  Raw:        'G7 Eng Frustration 2Level'
  Normalized: 'G7 2LD Eng Frustration'

  Raw:        'G7 Eng Instructional 2Level'
  Normalized: 'G7 2LD Eng Instructional'

  Raw:        'G7 Eng Independent 2Level'
  Normalized: 'G7 2LD Eng Independent'



In [8]:
# After normalization, check that KS3 columns match the canonical KS3 schema
df_ks3_norm = _normalize_philiri_columns(df_ks3_raw)
df_ks3_dedup = _drop_duplicates_columns(df_ks3_norm)

# How many canonical KS3 BoSY columns are found in the normalized file?
found = [c for c in KS3_BOSY_COLUMNS if c in df_ks3_dedup.columns]
missing = [c for c in KS3_BOSY_COLUMNS if c not in df_ks3_dedup.columns]
extra = [c for c in df_ks3_dedup.columns if c.startswith('G') and c not in KS3_BOSY_COLUMNS
         and not c.startswith(('G4','G5','G6'))  # not grade school columns
         and 'Total' not in c]

print(f'KS3 BoSY canonical columns: {len(KS3_BOSY_COLUMNS)}')
print(f'Found in normalized file:   {len(found)}')
print(f'Missing:                    {len(missing)}')
if missing:
    print(f'  {missing[:5]}')
print(f'Unexpected grade cols:      {len(extra)}')
if extra:
    print(f'  {extra[:5]}')

KS3 BoSY canonical columns: 64
Found in normalized file:   64
Missing:                    0
Unexpected grade cols:      0


---
## 3. Schema Deep-Dive: EoSY

EoSY only reassesses students who were at Frustration or Instructional level at BoSY. Students who were Grade Ready at BoSY are **automatically counted as Independent at EoSY** — they don't reappear individually in the raw data.

### 3a. KS2 EoSY (both years: same structure)

In [9]:
path_ks2_eosy_2425 = file_map[('ks2', '2024-25', 'EoSY')]
df_ks2_eosy = pd.read_csv(path_ks2_eosy_2425, nrows=0, low_memory=False)

print(f'KS2 EoSY 2024-25: {len(df_ks2_eosy.columns)} columns\n')
print('All columns:')
for c in df_ks2_eosy.columns:
    print(f'  {repr(c)}')

print(f'\nCanonical KS2 EoSY columns (data only): {len(KS2_EOSY_COLUMNS)}')
print('(= 3 grades × (1 Assessed + 2 langs × 3 levels) = 3 × 7 = 21)')

KS2 EoSY 2024-25: 36 columns

All columns:
  'Region'
  'Division'
  'District'
  'School ID'
  'School Name'
  'Total Enrolled (G4-G6)'
  'Total Assessed (G4-G6)'
  'Total Fil Frustration'
  'Total  Fil Instructional'
  'Total Fil Independent'
  'Total Fil Assessed'
  'Total Eng Frustration'
  'Total Eng Instructional'
  'Total Eng Independent'
  'Total Eng Assessed'
  'G4 Assessed'
  'G5 Assessed'
  'G6 Assessed'
  'G4 Fil Frustration'
  'G4 Fil Instructional'
  'G4 Eng Frustration'
  'G4 Eng Instructional'
  'G4 Eng Independent'
  'G4 Fil Independent'
  'G5 Fil Frustration'
  'G5 Eng Frustration'
  'G5 Fil Instructional'
  'G5 Fil Independent'
  'G5 Eng Instructional'
  'G5 Eng Independent'
  'G6 Fil Frustration'
  'G6 Fil Instructional'
  'G6 Fil Independent'
  'G6 Eng Frustration'
  'G6 Eng Instructional'
  'G6 Eng Independent'

Canonical KS2 EoSY columns (data only): 21
(= 3 grades × (1 Assessed + 2 langs × 3 levels) = 3 × 7 = 21)


In [10]:
# Note: EoSY has no 'Grade Ready' column — that category collapses into Independent
print('EoSY schema vs BoSY schema comparison (G4 Fil):\n')
print('BoSY G4 Fil columns:')
for c in [c for c in KS2_BOSY_COLUMNS if c.startswith('G4') and 'Fil' in c]:
    print(f'  {c}')
print()
print('EoSY G4 Fil columns:')
for c in [c for c in KS2_EOSY_COLUMNS if c.startswith('G4') and 'Fil' in c]:
    print(f'  {c}')
print()
print('Key difference: EoSY has no Grade Ready column. BoSY"s Grade Ready → EoSY"s Independent.')
print('This means the EoSY Independent count combines:')
print('  (a) BoSY Grade Ready students (automatic pass-through)')
print('  (b) Students who moved from 2LD/3LD to Independent during the year')

EoSY schema vs BoSY schema comparison (G4 Fil):

BoSY G4 Fil columns:
  G4 Fil Grade Ready
  G4 2LD Fil Frustration
  G4 2LD Fil Instructional
  G4 2LD Fil Independent
  G4 3LD Fil Frustration
  G4 3LD Fil Instructional
  G4 3LD Fil Independent

EoSY G4 Fil columns:
  G4 Fil Frustration
  G4 Fil Instructional
  G4 Fil Independent

Key difference: EoSY has no Grade Ready column. BoSY"s Grade Ready → EoSY"s Independent.
This means the EoSY Independent count combines:
  (a) BoSY Grade Ready students (automatic pass-through)
  (b) Students who moved from 2LD/3LD to Independent during the year


### 3b. KS3 EoSY: Schema difference between 2024-25 and 2025-26

This is the most significant structural difference across years. The 2024-25 KS3 EoSY has one `{G} Assessed` column per grade (combined), while 2025-26 splits to `{G} Fil Assessed` and `{G} Eng Assessed`.

In [11]:
# Compare KS3 EoSY column lists side by side
path_ks3_eosy_2425 = file_map[('ks3', '2024-25', 'EoSY')]
path_ks3_eosy_2526 = file_map[('ks3', '2025-26', 'EoSY')]

df_2425 = pd.read_csv(path_ks3_eosy_2425, nrows=0, low_memory=False)
df_2526 = pd.read_csv(path_ks3_eosy_2526, nrows=0, low_memory=False)

print(f'KS3 EoSY 2024-25: {len(df_2425.columns)} columns')
print(f'KS3 EoSY 2025-26: {len(df_2526.columns)} columns')
print()

# Show G7 columns from both
g7_2425 = [c for c in df_2425.columns if c.startswith('G7')]
g7_2526 = [c for c in df_2526.columns if c.startswith('G7')]

print('G7 columns in 2024-25 EoSY:')
for c in g7_2425:
    print(f'  {repr(c)}')

print()
print('G7 columns in 2025-26 EoSY:')
for c in g7_2526:
    print(f'  {repr(c)}')

print()
print('Notice: 2024-25 has "G7 Assessed" (one per grade), 2025-26 has "G7 Fil Assessed" + "G7 Eng Assessed"')

KS3 EoSY 2024-25: 43 columns
KS3 EoSY 2025-26: 45 columns

G7 columns in 2024-25 EoSY:
  'G7 Assessed'
  'G7 Fil Frustration'
  'G7 Fil Instructional'
  'G7 Fil Independent'
  'G7 Eng Frustration'
  'G7 Eng Instructional'
  'G7 Eng Independent'

G7 columns in 2025-26 EoSY:
  'G7 Fil Assessed'
  'G7 Fil Frustration'
  'G7 Fil Instructional'
  'G7 Fil Independent'
  'G7 Eng Assessed'
  'G7 Eng Frustration'
  'G7 Eng Instructional'
  'G7 Eng Independent'

Notice: 2024-25 has "G7 Assessed" (one per grade), 2025-26 has "G7 Fil Assessed" + "G7 Eng Assessed"


In [12]:
# ISSUE: KS3 EoSY 2024-25 has wrong Total Enrolled header
# The header says "Total Enrolled (G4-G6)" but the file is secondary (G7-G10).
# This is a copy-paste error in the Looker Studio report.
print('ISSUE: KS3 EoSY 2024-25 header bug\n')
print('Raw headers (first 7 columns):')
for c in df_2425.columns[:7]:
    print(f'  {repr(c)}')
print()
print('Expected: "Total Enrolled (G7-G10)" but found: "Total Enrolled (G4-G6)"')
print('Fix: _fix_ks3_eosy_2024_enrolled_header() renames it to (G7-G10).')
print('This only affects the metadata column name, not the data values.')

ISSUE: KS3 EoSY 2024-25 header bug

Raw headers (first 7 columns):
  'Region'
  'Division'
  'District'
  'School ID'
  'School Name'
  'Total Enrolled (G4-G6)'
  'Total Assessed (G4-G6)'

Expected: "Total Enrolled (G7-G10)" but found: "Total Enrolled (G4-G6)"
Fix: _fix_ks3_eosy_2024_enrolled_header() renames it to (G7-G10).
This only affects the metadata column name, not the data values.


---
## 4. Numeric Cleaning

All count columns are stored as strings with comma formatting in the raw CSVs (e.g., `"1,234"`). We verify the cleaning works correctly.

In [13]:
# Check for comma-formatted numbers in bronze
df_ks2_bosy_full = pd.read_csv(path_ks2_bosy_2425, nrows=20, low_memory=False)

print('Sample values from KS2 BoSY 2024-25 (first data column):')
first_data_col = KS2_BOSY_COLUMNS[0]  # 'G4 Assessed'
if first_data_col in df_ks2_bosy_full.columns:
    samples = df_ks2_bosy_full[first_data_col].head(5).tolist()
    print(f'  Column: {first_data_col}')
    print(f'  Raw values: {samples}')
    print(f'  dtype: {df_ks2_bosy_full[first_data_col].dtype}')
    comma_count = df_ks2_bosy_full[first_data_col].astype(str).str.contains(',').sum()
    print(f'  Rows with comma-formatted numbers: {comma_count}')
else:
    print(f'  Column {first_data_col} not found in raw; checking Total...')
    # Try total enrolled
    tot_col = [c for c in df_ks2_bosy_full.columns if 'Total Enrolled' in str(c)]
    if tot_col:
        samples = df_ks2_bosy_full[tot_col[0]].head(5).tolist()
        print(f'  Column: {tot_col[0]}, values: {samples}')

Sample values from KS2 BoSY 2024-25 (first data column):
  Column: G4 Assessed
  Raw values: ['1,829', '1,059', '884', '1,527', '1,380']
  dtype: object
  Rows with comma-formatted numbers: 11


In [14]:
# Verify numeric cleaning: silver should have float64 dtype for data columns
silver_dir = PROJECT_ROOT / 'data/silver/philiri'

for key in [('ks2','2024-25','BoSY'), ('ks3','2024-25','BoSY')]:
    ks, sy, period = key
    fpath = silver_dir / f'{ks}_{sy}_{period}.parquet'
    if not fpath.exists():
        print(f'MISSING: {fpath}')
        continue
    df = pd.read_parquet(fpath)
    data_cols = KS2_BOSY_COLUMNS if ks == 'ks2' else KS3_BOSY_COLUMNS
    present_cols = [c for c in data_cols if c in df.columns]
    dtypes = df[present_cols[:5]].dtypes.unique()
    print(f'{ks} {sy} {period}: data column dtypes = {list(dtypes)}')
    
    # Check for any string values that slipped through
    has_strings = any(df[c].dtype == object for c in present_cols)
    print(f'  Any string-dtype columns: {has_strings}')

ks2 2024-25 BoSY: data column dtypes = [dtype('int64'), dtype('float64')]
  Any string-dtype columns: False
ks3 2024-25 BoSY: data column dtypes = [dtype('int64')]
  Any string-dtype columns: False


---
## 5. Silver Layer Verification

For each of the 8 silver parquets we check:
- School count
- Column count (matches expected canonical schema)
- Missing canonical columns (expected NaN-fill for absent data)
- Sample row consistency with bronze

In [15]:
rows = []
for (ks, sy, period), path in sorted(file_map.items()):
    fpath = silver_dir / f'{ks}_{sy}_{period}.parquet'
    if not fpath.exists():
        rows.append({'Key': f'{ks} {sy} {period}', 'Status': 'MISSING'})
        continue
    df = pd.read_parquet(fpath)
    
    # Expected data columns
    if period == 'BoSY':
        expected_data = KS2_BOSY_COLUMNS if ks == 'ks2' else KS3_BOSY_COLUMNS
    else:
        if ks == 'ks2':
            expected_data = KS2_EOSY_COLUMNS
        else:
            expected_data = KS3_EOSY_COLUMNS_PER_LANG if sy == '2025-26' else KS3_EOSY_COLUMNS_PER_GRADE
    
    present = sum(1 for c in expected_data if c in df.columns)
    nan_only = sum(1 for c in expected_data if c in df.columns and df[c].isna().all())
    
    rows.append({
        'Key': f'{ks} {sy} {period}',
        'Schools': len(df),
        'Expected data cols': len(expected_data),
        'Present': present,
        'All-NaN cols': nan_only,
        'Tags present': all(c in df.columns for c in ['school_year','period','ks']),
    })

pd.DataFrame(rows).set_index('Key')

,Schools,Expected data cols,Present,All-NaN cols,Tags present
Key,,,,,
ks2 2024-25 BoSY,38525,45,45,0,True
ks2 2024-25 EoSY,38535,21,21,0,True
ks2 2025-26 BoSY,38525,45,45,0,True
ks2 2025-26 EoSY,38535,21,21,0,True
ks3 2024-25 BoSY,9599,64,64,0,True
ks3 2024-25 EoSY,10080,28,28,0,True
ks3 2025-26 BoSY,10897,64,64,0,True
ks3 2025-26 EoSY,10870,32,32,0,True


In [16]:
# Spot-check: load silver KS2 BoSY 2024-25 and compare one school against bronze
df_s = pd.read_parquet(silver_dir / 'ks2_2024-25_BoSY.parquet')
df_b = pd.read_csv(path_ks2_bosy_2425, low_memory=False)

# Clean bronze School ID for matching
df_b['_sid'] = pd.to_numeric(df_b['School ID'].astype(str).str.replace(',',''), errors='coerce').dropna().astype(int)
bronze_ids = set(df_b['_sid'].dropna().astype(int))
common = df_s.index.intersection(list(bronze_ids))
if len(common) > 0:
    check_id = common[0]
    b_row = df_b[df_b['_sid'] == check_id].iloc[0]
    s_row = df_s.loc[check_id]
    
    print(f'School ID: {check_id}')
    print(f'School Name (silver): {s_row.get("School Name", "N/A")}')
    print()
    print('G4 Assessed comparison:')
    col = 'G4 Assessed'
    print(f'  Bronze: {repr(b_row.get(col, "N/A"))}')
    print(f'  Silver: {s_row.get(col, "N/A")}')
    print()
    print('G4 2LD Fil Frustration comparison:')
    col2 = 'G4 2LD Fil Frustration'
    print(f'  Bronze: {repr(b_row.get(col2, "N/A"))}')
    print(f'  Silver: {s_row.get(col2, "N/A")}')

School ID: 136602
School Name (silver): Andres Bonifacio Elementary School

G4 Assessed comparison:
  Bronze: '1,829'
  Silver: 1829

G4 2LD Fil Frustration comparison:
  Bronze: np.int64(0)
  Silver: 0


---
## 6. School Counts and Coverage

How many schools appear in each file? Are the KS2 and KS3 populations largely disjoint?

In [17]:
print('School counts per silver parquet:\n')
silver_ids = {}
for (ks, sy, period), _ in sorted(file_map.items()):
    fpath = silver_dir / f'{ks}_{sy}_{period}.parquet'
    if fpath.exists():
        df = pd.read_parquet(fpath, columns=['school_year'])
        silver_ids[(ks, sy, period)] = set(df.index.tolist())
        print(f'  {ks:4s}  {sy}  {period:4s}: {len(df):,} schools')

School counts per silver parquet:

  ks2   2024-25  BoSY: 38,525 schools
  ks2   2024-25  EoSY: 38,535 schools
  ks2   2025-26  BoSY: 38,525 schools
  ks2   2025-26  EoSY: 38,535 schools
  ks3   2024-25  BoSY: 9,599 schools
  ks3   2024-25  EoSY: 10,080 schools
  ks3   2025-26  BoSY: 10,897 schools
  ks3   2025-26  EoSY: 10,870 schools


In [18]:
# KS2 vs KS3 overlap — should be near zero (elementary vs secondary schools)
ks2_ids_all = set()
ks3_ids_all = set()
for (ks, sy, period), ids in silver_ids.items():
    if ks == 'ks2':
        ks2_ids_all |= ids
    else:
        ks3_ids_all |= ids

overlap = ks2_ids_all & ks3_ids_all
print(f'Unique KS2 school IDs (across all timepoints): {len(ks2_ids_all):,}')
print(f'Unique KS3 school IDs (across all timepoints): {len(ks3_ids_all):,}')
print(f'Overlap (same School ID in both KS2 and KS3): {len(overlap):,}')
if overlap:
    print(f'  Sample overlap IDs: {list(overlap)[:5]}')
    print('  Note: some schools may have both elementary and secondary buildings under the same ID.')

Unique KS2 school IDs (across all timepoints): 38,796
Unique KS3 school IDs (across all timepoints): 11,004
Overlap (same School ID in both KS2 and KS3): 2,681
  Sample overlap IDs: [500000, 500001, 500002, 500003, 500004]
  Note: some schools may have both elementary and secondary buildings under the same ID.


In [19]:
# Within-KS consistency: are BoSY and EoSY school counts similar within each year?
print('School count comparison BoSY vs EoSY (same key stage and school year):\n')
for ks in ['ks2', 'ks3']:
    for sy in ['2024-25', '2025-26']:
        b_key = (ks, sy, 'BoSY')
        e_key = (ks, sy, 'EoSY')
        if b_key in silver_ids and e_key in silver_ids:
            b_ids = silver_ids[b_key]
            e_ids = silver_ids[e_key]
            in_both = b_ids & e_ids
            only_bosy = b_ids - e_ids
            only_eosy = e_ids - b_ids
            print(f'  {ks} {sy}: BoSY={len(b_ids):,} | EoSY={len(e_ids):,} | '
                  f'in both={len(in_both):,} | only BoSY={len(only_bosy):,} | only EoSY={len(only_eosy):,}')

School count comparison BoSY vs EoSY (same key stage and school year):

  ks2 2024-25: BoSY=38,525 | EoSY=38,535 | in both=38,264 | only BoSY=261 | only EoSY=271
  ks2 2025-26: BoSY=38,525 | EoSY=38,535 | in both=38,264 | only BoSY=261 | only EoSY=271
  ks3 2024-25: BoSY=9,599 | EoSY=10,080 | in both=9,410 | only BoSY=189 | only EoSY=670
  ks3 2025-26: BoSY=10,897 | EoSY=10,870 | in both=10,778 | only BoSY=119 | only EoSY=92


---
## 7. Sample Data Profiles

Let's look at actual data patterns for a sample school across BoSY and EoSY.

In [20]:
# Find a KS2 school present in both BoSY and EoSY 2024-25 with complete G4 data
df_ks2_bosy = pd.read_parquet(silver_dir / 'ks2_2024-25_BoSY.parquet')
df_ks2_eosy = pd.read_parquet(silver_dir / 'ks2_2024-25_EoSY.parquet')

both_ks2 = df_ks2_bosy.index.intersection(df_ks2_eosy.index)

# Find one with complete G4 data at BoSY
g4_bosy_cols = [c for c in KS2_BOSY_COLUMNS if c.startswith('G4')]
has_g4_bosy = df_ks2_bosy.loc[both_ks2, g4_bosy_cols].notna().all(axis=1)

if has_g4_bosy.any():
    sample_id = has_g4_bosy[has_g4_bosy].index[0]
    print(f'Sample School ID: {sample_id}')
    print(f'School Name: {df_ks2_bosy.loc[sample_id, "School Name"] if "School Name" in df_ks2_bosy.columns else "N/A"}')
    print()
    
    print('G4 Fil profile (BoSY 2024-25):')
    print(f'  Grade Ready:       {df_ks2_bosy.loc[sample_id, "G4 Fil Grade Ready"]}')
    print(f'  2LD Frustration:   {df_ks2_bosy.loc[sample_id, "G4 2LD Fil Frustration"]}')
    print(f'  2LD Instructional: {df_ks2_bosy.loc[sample_id, "G4 2LD Fil Instructional"]}')
    print(f'  2LD Independent:   {df_ks2_bosy.loc[sample_id, "G4 2LD Fil Independent"]}')
    print(f'  3LD Frustration:   {df_ks2_bosy.loc[sample_id, "G4 3LD Fil Frustration"]}')
    print(f'  3LD Instructional: {df_ks2_bosy.loc[sample_id, "G4 3LD Fil Instructional"]}')
    print(f'  3LD Independent:   {df_ks2_bosy.loc[sample_id, "G4 3LD Fil Independent"]}')
    print()
    
    if sample_id in df_ks2_eosy.index:
        print('G4 Fil profile (EoSY 2024-25):')
        print(f'  Frustration:   {df_ks2_eosy.loc[sample_id, "G4 Fil Frustration"]}')
        print(f'  Instructional: {df_ks2_eosy.loc[sample_id, "G4 Fil Instructional"]}')
        print(f'  Independent:   {df_ks2_eosy.loc[sample_id, "G4 Fil Independent"]}')
        print(f'  (EoSY Independent = 2LD/3LD-Ind movers + BoSY Grade Ready pass-throughs)')

Sample School ID: 136602
School Name: Andres Bonifacio Elementary School

G4 Fil profile (BoSY 2024-25):
  Grade Ready:       1200.0
  2LD Frustration:   0
  2LD Instructional: 5
  2LD Independent:   455
  3LD Frustration:   26
  3LD Instructional: 35
  3LD Independent:   108

G4 Fil profile (EoSY 2024-25):
  Frustration:   0
  Instructional: 58
  Independent:   81
  (EoSY Independent = 2LD/3LD-Ind movers + BoSY Grade Ready pass-throughs)


---
## 8. Known Issues and Open Questions

This section lists known data quality issues and decisions that still need to be made.

In [21]:
print('ISSUE 1: KS3 EoSY 2024-25 has no per-language Assessed column')
print('  BoSY: G7 Fil Assessed + G7 Eng Assessed (separate)')
print('  EoSY 2024-25: G7 Assessed (combined)')
print('  EoSY 2025-26: G7 Fil Assessed + G7 Eng Assessed (separate again)')
print()
print('  Impact: computing EoSY percentage profiles for 2024-25 KS3 requires using')
print('  the combined "G7 Assessed" as the denominator for BOTH Fil and Eng groups.')
print('  This is an approximation — the true per-language denominator is unknown.')
print()
print('  Decision (deferred): this issue only matters when computing EoSY percentages')
print('  for the gold layer. For now, silver stores raw counts as-is.')

ISSUE 1: KS3 EoSY 2024-25 has no per-language Assessed column
  BoSY: G7 Fil Assessed + G7 Eng Assessed (separate)
  EoSY 2024-25: G7 Assessed (combined)
  EoSY 2025-26: G7 Fil Assessed + G7 Eng Assessed (separate again)

  Impact: computing EoSY percentage profiles for 2024-25 KS3 requires using
  the combined "G7 Assessed" as the denominator for BOTH Fil and Eng groups.
  This is an approximation — the true per-language denominator is unknown.

  Decision (deferred): this issue only matters when computing EoSY percentages
  for the gold layer. For now, silver stores raw counts as-is.


In [22]:
print('ISSUE 2: The BoSY-only "levels down" features (3LD vs 2LD) cannot be linked to EoSY')
print('  BoSY grades a student as: Grade Ready, 2LD (F/I/Ind), or 3LD (F/I/Ind)')
print('  EoSY grades a student as: Frustration, Instructional, or Independent')
print('  There is no direct student-level link between the two assessments in the data.')
print()
print('  What we CAN do with BoSY-only features:')
print('  - Characterize a school\'s STARTING distribution across severity levels')
print('  - Use the 3LD:2LD ratio as a "depth of need" indicator')
print('  - Compute the proportion of Grade Ready at BoSY as a proxy for prior achievement')
print()
print('  Example: a school with 60% of G4 at 3LD Fil Frustration at BoSY is')
print('  deeper in need than one with 60% at 2LD Fil Frustration.')

ISSUE 2: The BoSY-only "levels down" features (3LD vs 2LD) cannot be linked to EoSY
  BoSY grades a student as: Grade Ready, 2LD (F/I/Ind), or 3LD (F/I/Ind)
  EoSY grades a student as: Frustration, Instructional, or Independent
  There is no direct student-level link between the two assessments in the data.

  What we CAN do with BoSY-only features:
  - Characterize a school's STARTING distribution across severity levels
  - Use the 3LD:2LD ratio as a "depth of need" indicator
  - Compute the proportion of Grade Ready at BoSY as a proxy for prior achievement

  Example: a school with 60% of G4 at 3LD Fil Frustration at BoSY is
  deeper in need than one with 60% at 2LD Fil Frustration.


In [23]:
print('ISSUE 3: BoSY Assessed vs sum of all BoSY categories')
print('  BoSY has: Grade Ready + 2LD (F+I+Ind) + 3LD (F+I+Ind) = 7 categories per lang')
print('  But: Assessed ≠ sum of these 7 categories in all cases')
print()
print('Checking for KS2 BoSY 2024-25 (G4 Fil):')
df_b = pd.read_parquet(silver_dir / 'ks2_2024-25_BoSY.parquet')
fil_cats = [
    'G4 Fil Grade Ready',
    'G4 2LD Fil Frustration', 'G4 2LD Fil Instructional', 'G4 2LD Fil Independent',
    'G4 3LD Fil Frustration', 'G4 3LD Fil Instructional', 'G4 3LD Fil Independent',
]
if 'G4 Assessed' in df_b.columns and all(c in df_b.columns for c in fil_cats):
    cat_sum = df_b[fil_cats].sum(axis=1)
    assessed = df_b['G4 Assessed']
    # Compare schools where both are non-NaN and non-zero
    both = (cat_sum > 0) & (assessed > 0) & cat_sum.notna() & assessed.notna()
    diff = (cat_sum[both] - assessed[both]).abs()
    print(f'  Schools with complete G4 Fil data: {both.sum():,}')
    print(f'  Schools where cat_sum == Assessed: {(diff < 0.5).sum():,}')
    print(f'  Max absolute difference: {diff.max():.0f}')
    print(f'  Median difference: {diff.median():.1f}')
    print()
    print('  Note: G4 Assessed may count students assessed in EITHER Fil or Eng,')
    print('  not just Fil. The category sum for Fil alone may be less than Assessed.')

ISSUE 3: BoSY Assessed vs sum of all BoSY categories
  BoSY has: Grade Ready + 2LD (F+I+Ind) + 3LD (F+I+Ind) = 7 categories per lang
  But: Assessed ≠ sum of these 7 categories in all cases

Checking for KS2 BoSY 2024-25 (G4 Fil):
  Schools with complete G4 Fil data: 38,475
  Schools where cat_sum == Assessed: 37,784
  Max absolute difference: 194
  Median difference: 0.0

  Note: G4 Assessed may count students assessed in EITHER Fil or Eng,
  not just Fil. The category sum for Fil alone may be less than Assessed.


In [24]:
print('ISSUE 4: Schools with all-zero or all-NaN data in some columns')
print()
for (ks, sy, period), _ in sorted(file_map.items()):
    fpath = silver_dir / f'{ks}_{sy}_{period}.parquet'
    if not fpath.exists():
        continue
    df = pd.read_parquet(fpath)
    if period == 'BoSY':
        data_cols = KS2_BOSY_COLUMNS if ks == 'ks2' else KS3_BOSY_COLUMNS
    else:
        data_cols = (KS2_EOSY_COLUMNS if ks == 'ks2' else
                     (KS3_EOSY_COLUMNS_PER_LANG if sy == '2025-26' else KS3_EOSY_COLUMNS_PER_GRADE))
    present_cols = [c for c in data_cols if c in df.columns]
    if not present_cols:
        continue
    all_nan = df[present_cols].isna().all(axis=1).sum()
    all_zero = (df[present_cols].fillna(0) == 0).all(axis=1).sum()
    print(f'  {ks} {sy} {period}: {all_nan} all-NaN rows, {all_zero} all-zero rows (of {len(df):,})')

ISSUE 4: Schools with all-zero or all-NaN data in some columns



  ks2 2024-25 BoSY: 0 all-NaN rows, 0 all-zero rows (of 38,525)
  ks2 2024-25 EoSY: 0 all-NaN rows, 0 all-zero rows (of 38,535)


  ks2 2025-26 BoSY: 0 all-NaN rows, 0 all-zero rows (of 38,525)


  ks2 2025-26 EoSY: 0 all-NaN rows, 0 all-zero rows (of 38,535)
  ks3 2024-25 BoSY: 0 all-NaN rows, 0 all-zero rows (of 9,599)


  ks3 2024-25 EoSY: 0 all-NaN rows, 5 all-zero rows (of 10,080)
  ks3 2025-26 BoSY: 0 all-NaN rows, 0 all-zero rows (of 10,897)
  ks3 2025-26 EoSY: 0 all-NaN rows, 0 all-zero rows (of 10,870)


---
## 9. What Comes Next: Gold Integration for PhilIRI

The silver layer is complete. The next step is to build a `data/gold/philiri_school_timepoints.parquet` and `data/gold/philiri_school_segments.parquet` analogous to the CRLA gold files.

**Open design decisions before implementing PhilIRI gold:**

1. **BoSY percentage computation**: How should Grade Ready + 2LD + 3LD collapse into a common proficiency scale?
   - Option A: Direct mapping — treat 3LD Frustration as ordinal 1, ..., Grade Ready as ordinal 7. Gives a 7-point scale for BoSY.
   - Option B: Collapse to 4 levels — Frustration (3LD F), Developing (3LD I + 2LD F), Instructional (3LD Ind + 2LD I), Grade Ready/Independent (2LD Ind + GR). Aligned with EoSY 4-level scale.
   - Option C: Use only the EoSY-compatible 3-level scale at both endpoints (losing the BoSY depth-of-need granularity).

2. **KS3 EoSY 2024-25 denominator**: Use combined `G7 Assessed` for both Fil and Eng when computing percentages.

3. **Cross-year comparability**: BoSY 2024-25 and 2025-26 share the same schema (safe to compare). EoSY 2024-25 KS3 vs EoSY 2025-26 KS3 differ in how Assessed is reported.

4. **CRLA–PhilIRI alignment**: CRLA covers G1–G3 (primary), PhilIRI covers G4–G10 (intermediate through senior). They don't overlap in grade coverage, making them complementary rather than redundant.

In [25]:
# Final summary: silver parquet inventory
print('PhilIRI silver layer — final inventory:\n')
print(f'{"Parquet file":<40} {"Schools":>8} {"Data cols":>10}')
print('-' * 62)
for (ks, sy, period), _ in sorted(file_map.items()):
    fpath = silver_dir / f'{ks}_{sy}_{period}.parquet'
    if fpath.exists():
        df = pd.read_parquet(fpath)
        if period == 'BoSY':
            n_data = len(KS2_BOSY_COLUMNS if ks == 'ks2' else KS3_BOSY_COLUMNS)
        else:
            n_data = len(KS2_EOSY_COLUMNS if ks == 'ks2' else
                         (KS3_EOSY_COLUMNS_PER_LANG if sy == '2025-26' else KS3_EOSY_COLUMNS_PER_GRADE))
        fname = f'{ks}_{sy}_{period}.parquet'
        print(f'{fname:<40} {len(df):>8,} {n_data:>10}')
print()
print('All 8 files present and populated. Gold layer pending design decisions above.')

PhilIRI silver layer — final inventory:

Parquet file                              Schools  Data cols
--------------------------------------------------------------
ks2_2024-25_BoSY.parquet                   38,525         45
ks2_2024-25_EoSY.parquet                   38,535         21


ks2_2025-26_BoSY.parquet                   38,525         45


ks2_2025-26_EoSY.parquet                   38,535         21
ks3_2024-25_BoSY.parquet                    9,599         64


ks3_2024-25_EoSY.parquet                   10,080         28
ks3_2025-26_BoSY.parquet                   10,897         64


ks3_2025-26_EoSY.parquet                   10,870         32

All 8 files present and populated. Gold layer pending design decisions above.
